In [ ]:
import numpy as np
import sympy as sp
import math
from scipy import special
import matplotlib.pyplot as plt
from pathlib import Path

plots_dir = Path('assets')
plots_dir.mkdir(parents=True, exist_ok=True)

print(f"The output folder for the plots is: {plots_dir.resolve()}")

g = sp.Symbol('g')

# The individual terms
t1 = ((2 * math.pi**2 -3)*g) / (6 * math.pi**3)
t2 = (-24 * special.zeta(3) + 5 - 4 * math.pi**2) / (32 * math.pi**4)
t3 = (11 + 2 * math.pi**2) / (256 * math.pi**5 * g)
t4 = (96 * special.zeta(3) + 75 + 8 * math.pi**2) / (4096 * math.pi**6 * g**2)
t5 = (3*(408*special.zeta(3) - 240 * special.zeta(5) + 213 + 14 * math.pi**2)) / (65536 * math.pi**7 * g**3)
t6 = (3 * (315 * special.zeta(3) - 240 * special.zeta(5) + 149 + 6 * math.pi ** 2)) / (65536 * math.pi ** 8 * g ** 4)

# Convert the SymPy expression into a fast Python function
fast_ratio65 = sp.lambdify(g, t6/t5, modules=['numpy', 'scipy'])
fast_ratio54 = sp.lambdify(g, t5/t4, modules=['numpy', 'scipy'])
fast_ratio43 = sp.lambdify(g, t4/t3, modules=['numpy', 'scipy'])
fast_ratio32 = sp.lambdify(g, t3/t2, modules=['numpy', 'scipy'])
fast_ratio21 = sp.lambdify(g, t2/t1, modules=['numpy', 'scipy'])

# Evaluate instantly for an array of 100 different g values
g_array = np.linspace(0.1, 3, 100)
results21 = fast_ratio21(g_array)
results32 = fast_ratio32(g_array)
results43 = fast_ratio43(g_array)
results54 = fast_ratio54(g_array)
results65 = fast_ratio65(g_array)

# Plot
plt.figure(figsize=(8, 5))
plt.plot(g_array, np.abs(results21), label=r'$|t_2 / t_1|$', color='blue', linewidth=2)
plt.plot(g_array, np.abs(results32), label=r'$|t_3 / t_2|$', color='orange', linewidth=2)
plt.plot(g_array, np.abs(results43), label=r'$|t_4 / t_3|$', color='green', linewidth=2)
plt.plot(g_array, np.abs(results54), label=r'$|t_5 / t_4|$', color='magenta', linewidth=2)
plt.plot(g_array, np.abs(results65), label=r'$|t_6 / t_5|$', color='red', linewidth=2)
plt.yscale('log')
plt.axhline(0.1, linestyle='--', color='gray')
plt.axvline(3, linestyle='--', color='gray')
plt.title('Comparison of absolutes values of ratios of the terms in $C(g)$ as a function of the coupling $g$', fontsize=14)
plt.xlabel('Value of $g$', fontsize=12)
plt.ylabel('Absolute values of ratios', fontsize=12)
plt.legend(fontsize=12)
plt.savefig(plots_dir / 'curvature_C_ratio_convergence.png', bbox_inches='tight')
plt.show()

In [ ]:
plots_dir = Path('assets')
plots_dir.mkdir(parents=True, exist_ok=True)
import matplotlib.pyplot as plt
import numpy as np

print(f"The output folder for the plots is: {plots_dir.resolve()}")

from spectrum_data import available_g_values
from identifiability import singular_values_sweep

g_values_for_sweep = available_g_values[1:]
sv_matrix= singular_values_sweep(g_values_for_sweep)
ten_numbers = np.linspace(0, 1, 10)
colours = plt.cm.viridis(ten_numbers)

for rank in range(10):
    plt.plot(
        g_values_for_sweep,
        sv_matrix[: ,rank],
        color = colours[rank],
        label = f'Rank {rank}',
    )

plt.xlabel('Values of the coupling constant, g.')
plt.ylabel('Singular Values of the Matrix')
plt.yscale('log')
plt.xscale('log')
plt.title('Sweeping over the values of the coupling constant, g.')
plt.savefig(plots_dir / 'identifiability_svd_sweep.png', bbox_inches='tight')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.show()

In [ ]:
plots_dir = Path('assets')
plots_dir.mkdir(parents=True, exist_ok=True)
import matplotlib.pyplot as plt
import numpy as np

print(f"The output folder for the plots is: {plots_dir.resolve()}")

from convex_baseline import solve_convex_problem, reconstruct_C_squared
from spectrum_data import available_g_values
from spline_basis import build_knot_vector
from ope_bounds_data import available_g_ope_values, ope_bounds_at_g

g_values_full = available_g_values[1:]
g_min = g_values_full[0]
g_max = g_values_full[-1]
degree = 3
knots = build_knot_vector(g_values_full[0], g_values_full[-1], 16, 3)
fitted_theta_best_lam = solve_convex_problem(g_values_full, knots, degree, lam = 0.01)
dense_grid = np.linspace(g_min, g_max, 200)

fitted_curves = {1: [], 2: [], 3: []}
lower_bounds = {1: [], 2: [], 3: []}
upper_bounds = {1: [], 2: [], 3: []}

for g in dense_grid:
    c_squared = reconstruct_C_squared(fitted_theta_best_lam, g, knots, degree)
    for state in [1, 2, 3]:
        fitted_curves[state].append(c_squared[state - 1])

for g in available_g_ope_values:
    for state in [1, 2, 3]:
        lower, upper = ope_bounds_at_g(state, g)
        lower_bounds[state].append(lower)
        upper_bounds[state].append(upper)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for idx, state in enumerate([1, 2, 3]):
    axes[idx].plot(dense_grid, fitted_curves[state], color='red', label=f'fitted')
    axes[idx].fill_between(available_g_ope_values, lower_bounds[state], upper_bounds[state],alpha=0.3, color= 'blue', label='known bounds')
    axes[idx].set_title(f'State {state}')
    axes[idx].set_xlabel('g')
    axes[idx].legend()

axes[0].set_ylabel('$C^2_n$')

plt.tight_layout()
plt.savefig(plots_dir / 'convex_baseline_vs_bounds.png', bbox_inches='tight')
plt.show()

In [ ]:
from convex_baseline import solve_convex_problem, reconstruct_C_squared
from spline_basis import build_knot_vector
from spectrum_data import available_g_values
from ope_bounds_data import ope_bounds_at_g, available_g_ope_values
import pandas as pd
import torch
from pinn_model import WilsonNetwork
import matplotlib.pyplot as plt

g_values = available_g_values[1:]
knots = build_knot_vector(g_values[0], g_values[-1], 16, 3)
theta = solve_convex_problem(g_values, knots, degree = 3, lam = 0.01)

rows = []

for g in available_g_ope_values:
    fitted = reconstruct_C_squared(theta, g, knots, degree = 3)
    lower_1, upper_1 = ope_bounds_at_g(1, g)
    lower_2, upper_2 = ope_bounds_at_g(2, g)
    lower_3, upper_3 = ope_bounds_at_g(3, g)
    rows.append({
        'g': g,
        'fitted_C1': fitted[0], 'lower_C1': lower_1, 'upper_C1': upper_1,
        'fitted_C2': fitted[1], 'lower_C2': lower_2, 'upper_C2': upper_2,
        'fitted_C3': fitted[2], 'lower_C3': lower_3, 'upper_C3': upper_3,
    })
df = pd.DataFrame(rows)

model = WilsonNetwork().double()
old_state_dict = torch.load('../models/trained_model_lam_0.pt')
remapped = {
    'hidden_layers.0.weight': old_state_dict['layer1.weight'],
    'hidden_layers.0.bias':   old_state_dict['layer1.bias'],
    'hidden_layers.1.weight': old_state_dict['layer2.weight'],
    'hidden_layers.1.bias':   old_state_dict['layer2.bias'],
    'hidden_layers.2.weight': old_state_dict['layer3.weight'],
    'hidden_layers.2.bias':   old_state_dict['layer3.bias'],
    'output_layer.weight':    old_state_dict['layer4.weight'],
    'output_layer.bias':      old_state_dict['layer4.bias'],
}
model.load_state_dict(remapped)
model.eval()

g_tensor = torch.tensor(available_g_ope_values, dtype=torch.float64).reshape(-1, 1)
with torch.no_grad():
    predictions = model(g_tensor)

pinn_C1 = predictions[:, 0].numpy()
pinn_C2 = predictions[:, 1].numpy()
pinn_C3 = predictions[:, 2].numpy()

df['pinn_C1'] = pinn_C1
df['pinn_C2'] = pinn_C2
df['pinn_C3'] = pinn_C3
df.to_csv('bounds_comparison_with_pinn.csv', index=False)

df['fitted_sum'] = df['fitted_C2'] + df['fitted_C3']
df['lower_sum'] = df['lower_C2'] + df['lower_C3']
df['upper_sum'] = df['upper_C2'] + df['upper_C3']
df['pinn_sum'] = df['pinn_C2'] + df['pinn_C3']

fig, axes = plt.subplots(2, 4, figsize=(17, 7.4), dpi=200)

panels = [
    (0, 'lower_C1', 'upper_C1', 'fitted_C1', 'pinn_C1', r'$C_1^2$'),
    (1, 'lower_C2', 'upper_C2', 'fitted_C2', 'pinn_C2', r'$C_2^2$'),
    (2, 'lower_C3', 'upper_C3', 'fitted_C3', 'pinn_C3', r'$C_3^2$'),
    (3, 'lower_sum', 'upper_sum', 'fitted_sum', 'pinn_sum', r'$C_2^2 + C_3^2$'),
]

row_configs = [
    (0, df, ""),
    (1, df[df['g'] <= 1.0], r", $g \leq 1$")
]

for row, data, title_suffix in row_configs:
    for col_idx, lo_col, hi_col, convex_col, pinn_col, base_title in panels:
        ax = axes[row, col_idx]
        ax.fill_between(data['g'], data[lo_col], data[hi_col], color='darkseagreen', alpha=0.55, label='known bounds', linewidth=0)
        ax.plot(data['g'], (data[lo_col] + data[hi_col]) / 2, color='seagreen', linewidth=1.2, linestyle=':', label='bound midpoint', zorder=1)
        ax.plot(data['g'], data[convex_col], color='royalblue', linewidth=1.8, label='convex baseline')
        ax.plot(data['g'], data[pinn_col], color='crimson', linewidth=1.8, linestyle='--', label='PINN')
        ax.set_xlabel('$g$')
        ax.set_title(base_title + title_suffix, fontsize=11)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
    axes[row, 0].set_ylabel(r"$C_n^2$")

axes[0, 0].legend(frameon=False, fontsize=8.5, loc='lower right')
fig.tight_layout()
fig.savefig('../notebooks/assets/four_panel_comparison.png', bbox_inches='tight')